# CSB & Weather Data Analysis

This notebook merges the processed Crop Sequence Boundaries (CSB) acreage data with aggregated weather data to explore relationships between climate variables and planting decisions.

In [ ]:
# Import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set plot style
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Define Paths
# Note: Adjust relative paths if running from a different directory
csb_path = "../data/merged_data/merged_CSB_total.csv"
weather_path = "../data/ny_weather_combined.feather"
HDD_path = "../data/HDD/"
CDD_city_path = HDD_path + "mctycddy.txt"
CDD_pop_w_path = HDD_path + "msacddy.txt"
HDD_city_path = HDD_path + "mctyhddy.txt"
HDD_pop_w_path = HDD_path + "msahddy.txt"

# 1. Load CSB Data
if os.path.exists(csb_path):
    csb_df = pd.read_csv(csb_path)
    print(f"CSB Data Loaded: {csb_df.shape}")
else:
    print(f"Error: CSB file not found at {csb_path}")

# 2. Load Weather Data
if os.path.exists(weather_path):
    weather_df = pd.read_feather(weather_path)
    print(f"Weather Data Loaded: {weather_df.shape}")
else:
    print(f"Error: Weather file not found at {weather_path}")

# 3. Load Degree Day Data
# Corrected specs to avoid overlapping columns and missing signs
msa_specs = [(1, 18), (18, 23), (23, 28), (28, 33), (33, 41), (41, 47), (47, 53), (53, 59), (59, 65)]
msa_names = ['STATE_REGION', 'MONTH_TOTAL', 'MON_DEV_FROM_NORM', 'MON_DEV_FROM_L_YR',
             'CUM_TOTAL', 'CUM_DEV_FROM_NORM', 'CUM_DEV_FROM_L_YR',
             'CUM_DEV_FROM_NORM_PRCT', 'CUM_DEV_FROM_L_YR_PRCT']

mcty_specs = [(0, 3), (3, 19), (19, 23), (23, 29), (29, 35), (35, 41), (41, 48), (48, 55), (55, 61), (61, 66), (66, 72)]
mcty_names = ['STATE', 'CITY', 'CALL', 'MONTH_TOTAL', 'MON_DEV_FROM_NORM', 'MON_DEV_FROM_L_YR',
              'CUM_TOTAL', 'CUM_DEV_FROM_NORM', 'CUM_DEV_FROM_L_YR',
              'CUM_DEV_FROM_NORM_PRCT', 'CUM_DEV_FROM_L_YR_PRCT']

def clean_and_convert_to_numeric(df, numeric_columns):
    for col in numeric_columns:
        if df[col].dtype == object:
            # Clean up trailing '-' separated by space, or any space-related parsing errors
            df[col] = df[col].astype(str).str.replace(r'-\s+', '-', regex=True).str.replace(r'\s+', '', regex=True)
        # Convert to numeric, turning invalid values (if any) into NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

numeric_cols_msa = ['MONTH_TOTAL', 'MON_DEV_FROM_NORM', 'MON_DEV_FROM_L_YR', 
                    'CUM_TOTAL', 'CUM_DEV_FROM_NORM', 'CUM_DEV_FROM_L_YR',
                    'CUM_DEV_FROM_NORM_PRCT', 'CUM_DEV_FROM_L_YR_PRCT']

numeric_cols_mcty = numeric_cols_msa

if os.path.exists(CDD_city_path):
    CDD_city_df = pd.read_fwf(CDD_city_path, skiprows=14, colspecs=mcty_specs, names=mcty_names)
    CDD_city_df = clean_and_convert_to_numeric(CDD_city_df, numeric_cols_mcty)
    print(f"CDD City Data Loaded: {CDD_city_df.shape}")
else:
    print(f"Error: CDD City file not found at {CDD_city_path}")

if os.path.exists(CDD_pop_w_path):
    CDD_pop_w_df = pd.read_fwf(CDD_pop_w_path, skiprows=15, colspecs=msa_specs, names=msa_names)
    CDD_pop_w_df = clean_and_convert_to_numeric(CDD_pop_w_df, numeric_cols_msa)
    print(f"CDD Pop W Data Loaded: {CDD_pop_w_df.shape}")
else:
    print(f"Error: CDD Pop W file not found at {CDD_pop_w_path}")

if os.path.exists(HDD_city_path):
    HDD_city_df = pd.read_fwf(HDD_city_path, skiprows=14, colspecs=mcty_specs, names=mcty_names)
    HDD_city_df = clean_and_convert_to_numeric(HDD_city_df, numeric_cols_mcty)
    print(f"HDD City Data Loaded: {HDD_city_df.shape}")
else:
    print(f"Error: HDD City file not found at {HDD_city_path}")

if os.path.exists(HDD_pop_w_path):
    HDD_pop_w_df = pd.read_fwf(HDD_pop_w_path, skiprows=15, colspecs=msa_specs, names=msa_names)
    HDD_pop_w_df = clean_and_convert_to_numeric(HDD_pop_w_df, numeric_cols_msa)
    print(f"HDD Pop W Data Loaded: {HDD_pop_w_df.shape}")
else:
    print(f"Error: HDD Pop W file not found at {HDD_pop_w_path}")

# 1. Data Exploration on Degrees Day Data

In [ ]:
# 1. Data Exploration on Degrees Day Data

# 1.1 Summary Statistics
print("--- CDD City Data Summary ---")
print(CDD_city_df.describe())
print(CDD_city_df.head())

print("\n--- CDD Population Weighted Data Summary ---")
print(CDD_pop_w_df.describe())

print("\n--- HDD City Data Summary ---")
print(HDD_city_df.describe())

print("\n--- HDD Population Weighted Data Summary ---")
print(HDD_pop_w_df.describe())

# 1.2 Visualizations

# Set up the figure for distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Distribution of Degree Days (City Data)', fontsize=16)

# Histograms for CDD
sns.histplot(data=CDD_city_df, x='MONTH_TOTAL', kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Monthly CDD (Cities)')
axes[0, 0].set_xlabel('Monthly Cooling Degree Days')

sns.histplot(data=CDD_city_df, x='CUM_TOTAL', kde=True, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution of Cumulative CDD (Cities)')
axes[0, 1].set_xlabel('Cumulative Cooling Degree Days')

# Histograms for HDD
sns.histplot(data=HDD_city_df, x='MONTH_TOTAL', kde=True, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Distribution of Monthly HDD (Cities)')
axes[1, 0].set_xlabel('Monthly Heating Degree Days')

sns.histplot(data=HDD_city_df, x='CUM_TOTAL', kde=True, ax=axes[1, 1], color='firebrick')
axes[1, 1].set_title('Distribution of Cumulative HDD (Cities)')
axes[1, 1].set_xlabel('Cumulative Heating Degree Days')

plt.tight_layout()
plt.show()

# histogram after log transformation
log_CDD_city_df = np.log(CDD_city_df + 1)
log_HDD_city_df = np.log(HDD_city_df + 1)
log_CDD_pop_w_df = np.log(CDD_pop_w_df + 1)
log_HDD_pop_w_df = np.log(HDD_pop_w_df + 1)

# Set up the figure for distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Distribution of Degree Days after Log Transformation (City Data)', fontsize=16)

# Histograms for CDD
sns.histplot(data=log_CDD_city_df, x='MONTH_TOTAL', kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Monthly CDD after Log Transformation (Cities)')
axes[0, 0].set_xlabel('Monthly Cooling Degree Days')

sns.histplot(data=log_CDD_city_df, x='CUM_TOTAL', kde=True, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution of Cumulative CDD after Log Transformation (Cities)')
axes[0, 1].set_xlabel('Cumulative Cooling Degree Days')

# Histograms for HDD
sns.histplot(data=log_HDD_city_df, x='MONTH_TOTAL', kde=True, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Distribution of Monthly HDD after Log Transformation (Cities)')
axes[1, 0].set_xlabel('Monthly Heating Degree Days')

sns.histplot(data=log_HDD_city_df, x='CUM_TOTAL', kde=True, ax=axes[1, 1], color='firebrick')
axes[1, 1].set_title('Distribution of Cumulative HDD after Log Transformation (Cities)')
axes[1, 1].set_xlabel('Cumulative Heating Degree Days')

plt.tight_layout()
plt.show()

# 1.3 Correlation Analysis
# Select numeric columns for correlation
numeric_cols = ['MONTH_TOTAL', 'MON_DEV_FROM_NORM', 'MON_DEV_FROM_L_YR', 
                'CUM_TOTAL', 'CUM_DEV_FROM_NORM', 'CUM_DEV_FROM_L_YR']

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# CDD Correlation
cdd_corr = CDD_city_df[numeric_cols].corr()
sns.heatmap(cdd_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
axes[0].set_title('Correlation Matrix - CDD City Data')

# HDD Correlation
hdd_corr = HDD_city_df[numeric_cols].corr()
sns.heatmap(hdd_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1])
axes[1].set_title('Correlation Matrix - HDD City Data')

plt.tight_layout()
plt.show()

# 1.4 Comparative Analysis (Proxy for Time Series)
# Calculate Normal and Last Year values for comparison
# Normal = Actual - Dev_from_Norm
# Last Year = Actual - Dev_from_Last_Year

# Create a copy to avoid SettingWithCopy warnings
hdd_comp = HDD_city_df.copy()
hdd_comp['NORMAL_MONTH'] = hdd_comp['MONTH_TOTAL'] - hdd_comp['MON_DEV_FROM_NORM']
hdd_comp['LAST_YEAR_MONTH'] = hdd_comp['MONTH_TOTAL'] - hdd_comp['MON_DEV_FROM_L_YR']

# Select top 10 cities by HDD Month Total for visualization
top_hdd_cities = hdd_comp.nlargest(10, 'MONTH_TOTAL')

# Melt for plotting
hdd_melted = top_hdd_cities.melt(id_vars=['CITY', 'STATE'], 
                                 value_vars=['MONTH_TOTAL', 'NORMAL_MONTH', 'LAST_YEAR_MONTH'],
                                 var_name='Metric', value_name='Degree Days')

plt.figure(figsize=(14, 8))
sns.barplot(data=hdd_melted, x='CITY', y='Degree Days', hue='Metric', palette='viridis')
plt.title('Top 10 Cities by Monthly HDD: Current vs Normal vs Last Year')
plt.xticks(rotation=45)
plt.ylabel('Heating Degree Days')
plt.legend(title='Comparison')
plt.tight_layout()
plt.show()


In [ ]:
# # Export Combined Weather Data
# output_path = "../data/merged_data/CSB_weather_combined.feather"

# if 'weather_annual' in locals():
#     weather_annual.to_feather(output_path)
#     print(f"Data exported to {output_path}")
# elif 'seasonal_weather' in locals():
#     seasonal_weather.to_feather(output_path)
#     print(f"Data exported to {output_path}")
# else:
#     print("No weather dataframe found to export.")

1. Data mining
2. Feature Engineering
3. Model Development
4. Model Validation
